<a href="https://colab.research.google.com/github/jimmyGit538/coin-market-cap-project/blob/production/CMC_Quotes_Historical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup

from google.colab import auth
auth.authenticate_user()
print("Authenticated")

!pip -q install requests pandas google-cloud-bigquery db-dtypes

import time
import math
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from google.cloud import bigquery
from getpass import getpass

Authenticated


In [ ]:
# 1) Config
# =========================

PROJECT_ID = "coinmarketcapproject"
DATASET_ID = "crypto_raw"

# Source table: category x coin rows (may contain duplicates)
TABLE_CATEGORY_COINS = "category_top20_coins_29_raw"   # existing table

# Output table: historical quotes with category attached
TABLE_QUOTES_HIST = "quotes_historical_30d_29_raw_V4"  # making a new one to test

# log missing coins that didn't return history
TABLE_QUOTES_HIST_ERRORS = "quotes_historical_30d_29_errors"

BASE_URL_V3 = "https://pro-api.coinmarketcap.com"
HIST_URL = f"{BASE_URL_V3}/v3/cryptocurrency/quotes/historical"

CMC_API_KEY = getpass("Enter your CoinMarketCap API key: ").strip()

HEADERS = {
    "Accepts": "application/json",
    "X-CMC_PRO_API_KEY": CMC_API_KEY
}

bq_client = bigquery.Client(project=PROJECT_ID)

# Date range. Adjust as needed.
from datetime import datetime, timezone

TIME_START = datetime(2026, 1, 14, tzinfo=timezone.utc)
TIME_END   = datetime(2026, 1, 24, tzinfo=timezone.utc)


INTERVAL = "daily"
CONVERT = "USD"

# API batching: safe defaults
COIN_IDS_PER_CALL = 10
MAX_RETRIES = 5


Enter your CoinMarketCap API key: ··········


In [ ]:
# 2) Helpers
# =========================

def iso_z(dt: datetime) -> str:
    """Format datetime as ISO 8601 with Z suffix."""
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def write_df_to_bigquery(df: pd.DataFrame, table_name: str, write_disposition: str = "WRITE_APPEND") -> None:
    """Write a DataFrame to BigQuery."""
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        write_disposition=write_disposition,
        autodetect=True
    )
    job = bq_client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f"Wrote {len(df):,} rows to {table_id} ({write_disposition}) ✅")


def fetch_quotes_historical_by_id(coin_id_batch, time_start: datetime, time_end: datetime,
                                 interval="daily", convert="USD") -> dict:
    """Call /v3/cryptocurrency/quotes/historical for a batch of coin IDs."""
    expected_points = (time_end.date() - time_start.date()).days + 1

    params = {
        "id": ",".join(str(int(x)) for x in coin_id_batch),
        "time_start": iso_z(time_start),
        "time_end": iso_z(time_end),
        "interval": interval,
        "convert": convert,
        # This prevents the "default 10 points" problem:
        "count": expected_points,
        "aux": "price,volume,market_cap,circulating_supply,total_supply,quote_timestamp,search_interval"
        # NOTE: intentionally NOT using skip_invalid here to can detect missing coins
    }

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        r = requests.get(HIST_URL, headers=HEADERS, params=params, timeout=60)

        if r.status_code == 200:
            return r.json()

        if r.status_code == 429:
            wait = 60 * attempt
            print(f"429 rate limit. Waiting {wait}s... (attempt {attempt}/{MAX_RETRIES})")
            time.sleep(wait)
            continue

        last_err = f"{r.status_code} {r.text[:500]}"
        print("Request failed:", last_err)
        time.sleep(2 * attempt)

    raise RuntimeError(f"Failed after retries. Last error: {last_err}")


def flatten_historical_payload(payload: dict, convert="USD") -> list[dict]:
    """Flatten the v3 quotes/historical payload into row dicts."""
    rows = []
    data = payload.get("data") or {}

    # v3 often returns dict keyed by coin id string
    for coin_key, obj in data.items():
        coin_id = obj.get("id")
        symbol = obj.get("symbol")
        name = obj.get("name")

        quotes = obj.get("quotes") or []
        for q in quotes:
            quote_ts = q.get("timestamp") or q.get("quote_timestamp")  # just in case

            quote = (q.get("quote") or {}).get(convert) or {}
            rows.append({
                "coin_id": coin_id,
                "coin_symbol": symbol,
                "coin_name": name,
                "quote_timestamp": quote_ts,
                "price": quote.get("price"),
                "market_cap": quote.get("market_cap"),
                "volume_24h": quote.get("volume_24h"),
                "circulating_supply": q.get("circulating_supply"),
                "total_supply": q.get("total_supply"),
                "ingestion_timestamp_utc": pd.Timestamp.utcnow()
            })

    return rows


In [ ]:
# 3) Pull category x coin rows from BigQuery (raw)
#  All categories come from the category–coin mapping table in BigQuery.
# The historical process only reads from that table and does not re-fetch or
# change categories. =========================

sql = f"""
SELECT
  category_id,
  SAFE_CAST(coin_id AS INT64) AS coin_id,
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_CATEGORY_COINS}`
WHERE coin_id IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY category_id, coin_id ORDER BY ingestion_timestamp_utc DESC) = 1
"""

df_raw = bq_client.query(sql).to_dataframe()
print("Raw rows from category table:", len(df_raw))
print("Distinct categories:", df_raw["category_id"].nunique())
print("Distinct coin_ids:", df_raw["coin_id"].nunique())


Raw rows from category table: 2434
Distinct categories: 29
Distinct coin_ids: 1859


In [ ]:
# 4) Dedup + pick CURRENT top 20 UNIQUE coins per category (Python)
# ------------------------------------------------------------
# Deduplicate and select top 20 coins per category
#
# The category source table can contain multiple rows for the
# same coin within a category (due to how the API data is
# structured and flattened). If we rank rows directly, those
# duplicates can take up multiple spots in the "top 20" and
# result in fewer than 20 unique coins being selected.
#
# To avoid this, we:
# 1) Sort coins within each category by CoinMarketCap rank
# 2) Remove duplicate (category_id, coin_id) pairs
# 3) Select the top 20 UNIQUE coins per category
#
# This ensures the historical pull is based on the intended
# top 20 coins, not top 20 rows.
# =========================
#
# df_dedup = (
#    df_raw
#    .sort_values(["category_id", "coin_cmc_rank", "coin_id"])
#    .drop_duplicates(subset=["category_id", "coin_id"], keep="first")
#)
#
#df_top20 = (
#    df_dedup
#    .groupby("category_id", group_keys=False)
#    .head(20)
#    .copy()
#)

print(df_raw.groupby("category_id")["coin_id"].nunique().describe())
print(df_raw.groupby("category_id")["coin_id"].nunique().sort_values().head(10))

# Build mapping: coin_id -> categories (some coins appear in multiple categories)
coin_to_categories = (
    df_raw.groupby("coin_id")["category_id"]
    .apply(lambda s: sorted(set(s)))
    .to_dict()
)

unique_coin_ids = sorted(df_raw["coin_id"].astype(int).unique().tolist())
print("\nUnique coins to pull history for:", len(unique_coin_ids))




Coins selected per category (should be <= 20):
count     29.000000
mean      83.931034
std       33.793418
min       16.000000
25%       62.000000
50%       83.000000
75%      117.000000
max      125.000000
Name: coin_id, dtype: float64

Any categories < 20 coins (means source table had fewer valid unique coins):
category_id
604f274bebccdd50cd175fbb    16
6051a82d66fc1b42617d6dd0    30
6634dccba7b6f0637eec196a    34
6246aade491a5b4fe942fa3f    36
692b02c1c0b341673d681a21    42
63248a04694d2a40b403f244    58
604f2749ebccdd50cd175fb9    60
5fb62da404d1dd4c73744883    62
6051a82666fc1b42617d6dc8    62
604f2776ebccdd50cd175fdc    64
Name: coin_id, dtype: int64

Unique coins to pull history for: 1859


In [ ]:
# 5) Pull historical quotes once per unique coin_id (batched)
# =========================

all_rows = []
missing_log = []  # track coins missing from payload

num_batches = math.ceil(len(unique_coin_ids) / COIN_IDS_PER_CALL)
print(f"\nPulling history in {num_batches} batches of {COIN_IDS_PER_CALL}...")

for i in range(num_batches):
    batch = unique_coin_ids[i * COIN_IDS_PER_CALL : (i + 1) * COIN_IDS_PER_CALL]
    print(f"Batch {i+1}/{num_batches} coin_ids={batch[:3]}... (+{max(0,len(batch)-3)} more)")

    payload = fetch_quotes_historical_by_id(
        coin_id_batch=batch,
        time_start=TIME_START,
        time_end=TIME_END,
        interval=INTERVAL,
        convert=CONVERT
    )

    # Which coins actually came back with *quotes*?
    returned_with_quotes = set()
    data = payload.get("data") or {}
    for _, obj in data.items():
        coin_id = obj.get("id")
        if coin_id is not None:
            # If coin is in payload, but has no quotes, log it differently
            if not (obj.get("quotes") or []):
                missing_log.append({
                    "coin_id": int(coin_id),
                    "missing_reason": "Returned in payload but no quotes available",
                    "time_start": TIME_START.date().isoformat(),
                    "time_end": TIME_END.date().isoformat(),
                    "ingestion_timestamp_utc": pd.Timestamp.utcnow()
                })
            else:
                returned_with_quotes.add(int(coin_id))

    for cid in batch:
        if int(cid) not in returned_with_quotes and int(cid) not in [m['coin_id'] for m in missing_log if m['missing_reason'] == 'Returned in payload but no quotes available']:
            # Only log as 'Not returned in payload' if it wasn't in the data at all
            # or if it was in the data but already logged for no quotes.
            # This avoids double logging for coins with no quotes.
            if int(cid) not in [int(c) for c_key, c_obj in data.items() for c in [c_obj.get('id')] if c is not None]:
                 missing_log.append({
                    "coin_id": int(cid),
                    "missing_reason": "Not returned in payload",
                    "time_start": TIME_START.date().isoformat(),
                    "time_end": TIME_END.date().isoformat(),
                    "ingestion_timestamp_utc": pd.Timestamp.utcnow()
                })

    rows = flatten_historical_payload(payload, convert=CONVERT)
    all_rows.extend(rows)

    time.sleep(2) # Add a 2-second delay after each batch
    #this time is enough to avoid API 429 error issues

print("\nTotal historical rows pulled (coin-level):", len(all_rows))
df_hist = pd.DataFrame(all_rows)

# sanity check
if not df_hist.empty:
    df_hist["quote_date"] = pd.to_datetime(df_hist["quote_timestamp"]).dt.date
    print("Distinct days returned:", df_hist["quote_date"].nunique())
    print("Distinct coins returned:", df_hist["coin_id"].nunique())



Pulling history in 186 batches of 10...
Batch 1/186 coin_ids=[1, 2, 45]... (+7 more)
Batch 2/186 coin_ids=[372, 377, 512]... (+7 more)
Batch 3/186 coin_ids=[954, 1026, 1027]... (+7 more)
Batch 4/186 coin_ids=[1214, 1312, 1321]... (+7 more)
Batch 5/186 coin_ids=[1455, 1466, 1483]... (+7 more)
Batch 6/186 coin_ids=[1631, 1637, 1659]... (+7 more)
Batch 7/186 coin_ids=[1723, 1732, 1757]... (+7 more)
Batch 8/186 coin_ids=[1810, 1826, 1831]... (+7 more)
Batch 9/186 coin_ids=[1958, 1966, 1967]... (+7 more)
Batch 10/186 coin_ids=[2011, 2012, 2019]... (+7 more)
Batch 11/186 coin_ids=[2096, 2130, 2133]... (+7 more)
Batch 12/186 coin_ids=[2277, 2280, 2283]... (+7 more)
Batch 13/186 coin_ids=[2354, 2363, 2384]... (+7 more)
Batch 14/186 coin_ids=[2429, 2467, 2469]... (+7 more)
Batch 15/186 coin_ids=[2535, 2544, 2545]... (+7 more)
Batch 16/186 coin_ids=[2629, 2634, 2638]... (+7 more)
Batch 17/186 coin_ids=[2720, 2729, 2745]... (+7 more)
Batch 18/186 coin_ids=[2868, 2870, 2882]... (+7 more)
Batch 19

In [ ]:
# 6) Attach categories back to each coin’s quotes
# =========================

expanded = []
for _, r in df_hist.iterrows():
    cid = int(r["coin_id"])
    cats = coin_to_categories.get(cid, [])
    for cat_id in cats:
        row = r.to_dict()
        row["category_id"] = cat_id
        expanded.append(row)

df_hist_by_category = pd.DataFrame(expanded)
print("\nRows after attaching categories:", len(df_hist_by_category))
print("Distinct categories in final:", df_hist_by_category["category_id"].nunique())
print("Distinct coins in final:", df_hist_by_category["coin_id"].nunique())



Rows after attaching categories: 17940
Distinct categories in final: 29
Distinct coins in final: 1447


In [ ]:
# 7) Write outputs to BigQuery
# =========================

# Write historical output
if not df_hist_by_category.empty:
    write_df_to_bigquery(df_hist_by_category, TABLE_QUOTES_HIST, write_disposition="WRITE_APPEND")

# Write missing coin log
if missing_log:
    df_missing = pd.DataFrame(missing_log)
    print("\nMissing coin payload count:", len(df_missing))
    write_df_to_bigquery(df_missing, TABLE_QUOTES_HIST_ERRORS, write_disposition="WRITE_APPEND")
else:
    print("\nNo missing coins detected in payload ✅")

Wrote 17,940 rows to coinmarketcapproject.crypto_raw.quotes_historical_30d_29_raw_V4 (WRITE_APPEND) ✅

Missing coin payload count: 412
Wrote 412 rows to coinmarketcapproject.crypto_raw.quotes_historical_30d_29_errors (WRITE_APPEND) ✅


In [ ]:
# 8) QA summary
# =========================

print("\n===== QA SUMMARY =====")
print("Category rows:", len(df_raw))
print("Unique coins requested:", len(unique_coin_ids))
print("Coin-level historical rows pulled:", len(df_hist))
print("Category-attached historical rows:", len(df_hist_by_category))
if missing_log:
    print("Coins missing from payload:", len(pd.DataFrame(missing_log)["coin_id"].unique()))
else:
    print("Coins missing from payload: 0")


===== QA SUMMARY =====
Category rows: 2434
Unique coins requested: 1859
Coin-level historical rows pulled: 12966
Category-attached historical rows: 17940
Coins missing from payload: 412
